# Try ResuVL: read a CV page into JSON

ResuVL models read a CV page **image** and write its content as JSON: name, contact details, work, education,
skills, languages and more. They never use the PDF text layer, so scans and photos work the same way as PDFs.

This notebook downloads a released model, reads an example page with it, shows field by field what it got right
and wrong against a reviewed label, and lets you upload your own CV page. Nothing is trained here; to see how the
models are trained, open [train_on_eight_pages.ipynb](train_on_eight_pages.ipynb).

**Where to run it.** On Colab the default CPU runtime works; *Runtime > Change runtime type > T4 GPU* is faster
(times for both are given before each step). Locally: `pip install -e ".[notebook]"` and open this notebook from
the repository.

## 1. Setup

On Colab this clones the repository and installs the package (about 1 minute). Locally it only moves to the
repository root. `MODEL` picks the model: ResuVL-S by default; change it to `furkankucuk/ResuVL-L` for the most
accurate model.

In [ ]:
import logging
import os
import sys
import warnings
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("/content/tiny-resume-parser").exists():
        !git clone -q https://github.com/iamkucuk/tiny-resume-parser /content/tiny-resume-parser
    %cd /content/tiny-resume-parser
    %pip install -q -e .
elif Path.cwd().name == "examples":
    os.chdir("..")

os.environ["TINY_RESUME_PARSER_DATA"] = "examples"
sys.path.insert(0, "src")  # an editable install is only seen by a restarted kernel
sys.path.insert(0, "examples")

# Two known warnings that do not apply here: a tokenizer regex notice meant for another model family, and
# pin_memory on Apple GPUs. Every other warning still shows.
warnings.filterwarnings("ignore", message=".*pin_memory.*")
logging.getLogger("transformers.tokenization_utils_base").addFilter(lambda r: "fix_mistral_regex" not in r.getMessage())

MODEL = "furkankucuk/ResuVL-S"  # 0.26B, fastest. "furkankucuk/ResuVL-L" is the most accurate (0.87B, slower)
print("working in", Path.cwd(), "| model", MODEL)

## 2. Read one page from the command line

This is how the models are used in practice: `tiny-resume-parser parse FILE --model MODEL` reads every page of a PDF
or image file and prints one JSON record per page. `valid` says whether the answer passes the output schema. The
page is from the validation split, which no released model was trained on. The first run downloads the model.

In [ ]:
from IPython.display import display

from tiny_resume_parser.data import page_image, read_jsonl

train_rows = read_jsonl("examples/train.jsonl")
eval_rows = read_jsonl("examples/eval.jsonl")
row = eval_rows[0]
display(page_image(row).resize((496, 702)))
Path("runs/try").mkdir(parents=True, exist_ok=True)
!{sys.executable} -m tiny_resume_parser.parse examples/{row["image"]} --model {MODEL} --out runs/try/page.json
print(Path("runs/try/page.json").read_text()[:2500])

`valid: false` does not mean the answer is lost. It means the answer is JSON but some value breaks a rule of the
output schema (`src/tiny_resume_parser/schema.py`), for example a project type outside the allowed list. The record is
still returned, and its values are still scored. The next cell prints which rules a non-valid answer breaks.

In [ ]:
import json

from pydantic import ValidationError

from tiny_resume_parser.schema import CVRecord

page = json.loads(Path("runs/try/page.json").read_text())[0]
try:
    CVRecord.model_validate(page["record"])
    print("valid: the answer passes every schema rule")
except ValidationError as e:
    for err in e.errors():
        print(".".join(map(str, err["loc"])), "|", err["msg"], "| got", repr(err["input"]))

## 3. See why it scores what it scores

The explorer reads a page and puts every value of the answer next to the label's value:

- **Fields**: *match* (same letters and digits), *differs* (same field, a letter or digit differs, highlighted),
  *extra* (not in the label), *missing* (in the label, left out). Field F1 is computed from exactly these pairs.
- **Scores**: strict, lenient and content F1 for this page, each with its rule.
- **JSON**: the answer after repair, as `parse` returns it.
- **Model input**: the image tiles the model sees and how many tokens they become.

Pick a page or **upload your own** (PNG, JPEG or PDF; the first page is read) and press **Read page**. An uploaded
page has no label, so its values are listed without a status. The labels were written by a large model and
reviewed only for the 254 test and development pages, so a *differs* can also be a label error.

In [ ]:
from explorer import Explorer, report

explorer = Explorer(MODEL, train_rows, eval_rows, max_new_tokens=2000)
explorer.show()

Widgets do not render in a saved copy of a notebook (for example on GitHub). The next cell prints the scores
and the field table of the page shown above as plain HTML, so a saved copy still shows them.

In [ ]:
report(explorer)

## 4. Next steps

- The scores of all models on 204 test pages: [README](https://github.com/iamkucuk/tiny-resume-parser#results).
- How a page becomes JSON, step by step: [docs/how-it-works.md](https://github.com/iamkucuk/tiny-resume-parser/blob/main/docs/how-it-works.md).
- Train a model yourself on the eight example pages: [train_on_eight_pages.ipynb](https://colab.research.google.com/github/iamkucuk/tiny-resume-parser/blob/main/examples/train_on_eight_pages.ipynb).

On Colab, the next cell ends the session so a GPU is not held; skip it to keep working.

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import runtime

    runtime.unassign()